# Combined Anatomical (CT) Receptor Annotation: Symptoms + Cognition

In [ ]:
!pip install pingouin

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
from pingouin import multicomp

In [ ]:
matplotlib.rcParams.update(matplotlib.rcParamsDefault)
%matplotlib inline
plt.rcParams['font.size'] = 7
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "path"

In [ ]:
# Set paths (edit to match your local layout)
ep_anat_dir = Path("./outputs/anat")
cog_anat_dir = Path("./outputs/anat_cog")
figs_dir = Path("./figures")

## PET Map Definitions & Averaging

In [ ]:
mni152_pet_items = [
    ('aghourian2017', 'feobv', 'MNI152', '1mm'),
    ('alarkurtti2015', 'raclopride', 'MNI152', '3mm'),
    ('bedard2019', 'feobv', 'MNI152', '1mm'),
    ('beliveau2017', 'az10419369', 'MNI152', '1mm'),
    ('beliveau2017', 'cimbi36', 'MNI152', '1mm'),
    ('beliveau2017', 'cumi101', 'MNI152', '1mm'),
    ('beliveau2017', 'dasb', 'MNI152', '1mm'),
    ('beliveau2017', 'sb207145', 'MNI152', '1mm'),
    ('castrillon2023', 'cmrglc', 'MNI152', '3mm'),
    ('ding2010', 'mrb', 'MNI152', '1mm'),
    ('dubois2015', 'abp688', 'MNI152', '1mm'),
    ('dukart2018', 'flumazenil', 'MNI152', '3mm'),
    ('dukart2018', 'fpcit', 'MNI152', '3mm'),
    ('fazio2016', 'madam', 'MNI152', '3mm'),
    ('finnema2016', 'ucbj', 'MNI152', '1mm'),
    ('gallezot2010', 'p943', 'MNI152', '1mm'),
    ('gallezot2017', 'gsk189254', 'MNI152', '1mm'),
    ('galovic2021', 'ge179', 'MNI152', '1mm'),
    ('hesse2017', 'methylreboxetine', 'MNI152', '3mm'),
    ('hillmer2016', 'flubatine', 'MNI152', '1mm'),
    ('jaworska2020', 'fallypride', 'MNI152', '1mm'),
    ('kaller2017', 'sch23390', 'MNI152', '3mm'),
    ('kantonen2020', 'carfentanil', 'MNI152', '3mm'),
    ('kim2020', 'ps13', 'MNI152', '2mm'),
    ('laurikainen2018', 'fmpepd2', 'MNI152', '1mm'),
    ('lois2018', 'pbr28', 'MNI152', '2mm'),
    ('lukow2022', 'ro154513', 'MNI152', '2mm'),
    ('malen2022', 'raclopride', 'MNI152', '2mm'),
    ('naganawa2020', 'lsn3172176', 'MNI152', '1mm'),
    ('norgaard2021', 'flumazenil', 'MNI152', '1mm'),
    ('normandin2015', 'omar', 'MNI152', '1mm'),
    ('radnakrishnan2018', 'gsk215083', 'MNI152', '1mm'),
    ('rosaneto', 'abp688', 'MNI152', '1mm'),
    ('sandiego2015', 'flb457', 'MNI152', '1mm'),
    ('sasaki2012', 'fepe2i', 'MNI152', '1mm'),
    ('satterthwaite2014', 'meancbf', 'MNI152', '1mm'),
    ('savli2012', 'altanserin', 'MNI152', '3mm'),
    ('savli2012', 'dasb', 'MNI152', '3mm'),
    ('savli2012', 'p943', 'MNI152', '3mm'),
    ('savli2012', 'way100635', 'MNI152', '3mm'),
    ('smart2019', 'abp688', 'MNI152', '1mm'),
    ('smith2017', 'flb457', 'MNI152', '1mm'),
    ('tuominen', 'feobv', 'MNI152', '2mm'),
    ('turtonen2020', 'carfentanil', 'MNI152', '1mm'),
    ('vijay2018', 'ly2795050', 'MNI152', '2mm'),
    ('wey2016', 'martinostat', 'MNI152', '2mm'),
]

fstr_list = ["-".join(item) for item in mni152_pet_items]

fstr_reordered_list = [
    # serotonin
    ('beliveau2017-cumi101-MNI152-1mm', "5-HT1a"),
    ('savli2012-way100635-MNI152-3mm', "5-HT1a"),
    ('beliveau2017-az10419369-MNI152-1mm', "5-HT1b"),
    ('gallezot2010-p943-MNI152-1mm', "5-HT1b"),
    ('savli2012-p943-MNI152-3mm', "5-HT1b"),
    ('beliveau2017-cimbi36-MNI152-1mm', "5-HT2a"),
    ('savli2012-altanserin-MNI152-3mm', "5-HT2a"),
    ('beliveau2017-sb207145-MNI152-1mm', "5-HT4"),
    ('radnakrishnan2018-gsk215083-MNI152-1mm', "5-HT6"),
    ('beliveau2017-dasb-MNI152-1mm', "5-HTT"),
    ('fazio2016-madam-MNI152-3mm', "5-HTT"),
    ('savli2012-dasb-MNI152-3mm', "5-HTT"),
    # dopamine
    ('kaller2017-sch23390-MNI152-3mm', "D1"),
    ('alarkurtti2015-raclopride-MNI152-3mm', "D2"),
    ('jaworska2020-fallypride-MNI152-1mm', "D2"),
    ('malen2022-raclopride-MNI152-2mm', "D2"),
    ('sandiego2015-flb457-MNI152-1mm', "D2"),
    ('smith2017-flb457-MNI152-1mm', "D2"),
    ('dukart2018-fpcit-MNI152-3mm', "DAT"),
    ('sasaki2012-fepe2i-MNI152-1mm', "DAT"),
    # norepinephrine
    ('ding2010-mrb-MNI152-1mm', "NET"),
    ('hesse2017-methylreboxetine-MNI152-3mm', "NET"),
    # histamine
    ('gallezot2017-gsk189254-MNI152-1mm', "H3"),
    # acetylcholine
    ('aghourian2017-feobv-MNI152-1mm', "VAChT"),
    ('bedard2019-feobv-MNI152-1mm', "VAChT"),
    ('tuominen-feobv-MNI152-2mm', "VAChT"),
    ('hillmer2016-flubatine-MNI152-1mm', "a4b2"),
    ('naganawa2020-lsn3172176-MNI152-1mm', "M1"),
    # cannabinoid
    ('laurikainen2018-fmpepd2-MNI152-1mm', "CB1"),
    ('normandin2015-omar-MNI152-1mm', "CB1"),
    # opioid
    ('kantonen2020-carfentanil-MNI152-3mm', "MOR"),
    ('turtonen2020-carfentanil-MNI152-1mm', "MOR"),
    ('vijay2018-ly2795050-MNI152-2mm', "KOR"),
    # glutamate
    ('galovic2021-ge179-MNI152-1mm', "NMDA"),
    ('dubois2015-abp688-MNI152-1mm', "mGluR5"),
    ('rosaneto-abp688-MNI152-1mm', "mGluR5"),
    ('smart2019-abp688-MNI152-1mm', "mGluR5"),
    # gaba
    ('dukart2018-flumazenil-MNI152-3mm', "GABAa"),
    ('lukow2022-ro154513-MNI152-2mm', "GABAa"),
    ('norgaard2021-flumazenil-MNI152-1mm', "GABAa"),
    # excluded from averaging but kept for reference
    ('wey2016-martinostat-MNI152-2mm', "HDAC"),
    ('kim2020-ps13-MNI152-2mm', "COX-1"),
    ('finnema2016-ucbj-MNI152-1mm', "SV2A"),
    ('lois2018-pbr28-MNI152-2mm', "TSPO"),
    ('satterthwaite2014-meancbf-MNI152-1mm', "cbf"),
    ('castrillon2023-cmrglc-MNI152-3mm', "cmrglc"),
]

fstr_reordered_labels = [_[1] for _ in fstr_reordered_list]
fstr_reorder_idx = [fstr_list.index(_[0]) for _ in fstr_reordered_list]

In [ ]:
## Average PET maps by receptor/transporter type

map_to_receptor = {item[0]: item[1] for item in fstr_reordered_list}

mapinfo = pd.DataFrame({
    'name': fstr_list,
    'receptor': [map_to_receptor.get('-'.join(item), 'unknown') for item in mni152_pet_items]
})
mapinfo['orig_idx'] = range(len(mapinfo))

exclude_receptors = ['cbf', 'cmrglc', 'HDAC', 'SV2A', 'COX-1']
mapinfo_nt = mapinfo[~mapinfo['receptor'].isin(exclude_receptors)].copy()

# Load parcellated PET data (DK atlas, 68 cortical regions)
# These were saved by the neuromaps parcellation step
mni152_data = np.load(ep_anat_dir / "mni152_pet_items_data.npy")

# Average by receptor type (alphabetical)
unique_receptors_sorted = sorted(mapinfo_nt['receptor'].unique())
averaged_data_list = []

print("Averaging maps by receptor type:\n")
for receptor in unique_receptors_sorted:
    idx = mapinfo_nt[mapinfo_nt['receptor'] == receptor]['orig_idx'].values
    averaged_data_list.append(np.mean(mni152_data[idx], axis=0))
    map_names = mapinfo_nt[mapinfo_nt['receptor'] == receptor]['name'].tolist()
    print(f"{receptor}: averaged {len(idx)} map(s)")
    for name in map_names:
        print(f"    - {name}")

pet_maps_averaged = np.array(averaged_data_list)
pet_labels = unique_receptors_sorted
n_receptors = len(pet_labels)

print(f"\nFinal: {pet_maps_averaged.shape[0]} receptor types, {pet_maps_averaged.shape[1]} parcels")

## Load Precomputed Spatial Correlation Results

In [ ]:
# Symptom anat: shape (n_receptors, 4) — columns: pos, neg, gps, ymrs
# Note: file names are xload_vec_ctx_sr_* (not xload_mat_ctx_deg_sr_* like RSFC)
ep_sr_stat = np.load(ep_anat_dir / "xload_vec_ctx_sr_stat.npy")
ep_sr_pval = np.load(ep_anat_dir / "xload_vec_ctx_sr_pval.npy")
print(f"Symptom sr_stat shape: {ep_sr_stat.shape}")
print(f"Symptom sr_pval shape: {ep_sr_pval.shape}")

# Cognition anat: shape (n_receptors,)
cog_sr_stat = np.load(cog_anat_dir / "xload_vec_ctx_sr_stat.npy")
cog_sr_pval = np.load(cog_anat_dir / "xload_vec_ctx_sr_pval.npy")
print(f"Cognition sr_stat shape: {cog_sr_stat.shape}")
print(f"Cognition sr_pval shape: {cog_sr_pval.shape}")

## Combine: Positive, Negative, General, Cognition

In [ ]:
# Select pos=0, gps=2 from symptom array
sel_idx = [0, 2]
row_labels = ["Positive", "General", "Cognition"]

# Stack into (3, n_receptors)
sr_stat = np.vstack([ep_sr_stat[:, sel_idx].T, cog_sr_stat.reshape(1, -1)])
sr_pval = np.vstack([ep_sr_pval[:, sel_idx].T, cog_sr_pval.reshape(1, -1)])


print(f"Combined sr_stat shape: {sr_stat.shape}")
print(f"Combined sr_pval shape: {sr_pval.shape}")

In [ ]:
# Threshold rho at uncorrected p < 0.05
sr_stat_sig = sr_stat.copy()
sr_stat_sig[sr_pval > 0.05] = 0.0

# FDR correction per row (each domain independently)
sr_qval = np.zeros_like(sr_pval)
for i in range(len(row_labels)):
    _, qvals = multicomp(sr_pval[i, :], method='fdr_bh')
    sr_qval[i, :] = qvals

# Print summary
for i, label in enumerate(row_labels):
    n_sig_uncorr = np.sum(sr_pval[i, :] < 0.05)
    n_sig_fdr = np.sum(sr_qval[i, :] < 0.05)
    print(f"{label}: {n_sig_uncorr} sig (uncorrected), {n_sig_fdr} sig (FDR)")

## ACNP-Style Heatmaps

In [ ]:
def plot_heatmap(data, row_labels, col_labels, cmap, clim, cbar_label, title,
                 figsize=(12, 3), sig_mask=None, fname=None, figs_dir=None):
    """
    Generic heatmap for receptor annotation results.
    sig_mask: boolean array same shape as data — draws red border around True cells.
    """
    fig, ax = plt.subplots(figsize=figsize, layout="constrained")
    im = ax.imshow(data, cmap=cmap, vmin=clim[0], vmax=clim[1], aspect="equal")

    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=90, fontsize=10)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=12, fontweight='bold')

    ax.tick_params(axis='both', which='both', length=0)

    # Red borders for FDR-surviving cells
    if sig_mask is not None:
        for i in range(sig_mask.shape[0]):
            for j in range(sig_mask.shape[1]):
                if sig_mask[i, j]:
                    ax.add_patch(plt.Rectangle(
                        (j - 0.5, i - 0.5), 1, 1,
                        fill=False, edgecolor='red', linewidth=2
                    ))

    cbar = plt.colorbar(im, shrink=0.6, pad=0.02)
    cbar.set_label(cbar_label, fontsize=10)

    ax.set_title(title, fontsize=14, fontweight='bold')

    if fname and figs_dir:
        fig.savefig(figs_dir / fname, dpi=300, bbox_inches='tight')

    plt.show()
    return fig

In [ ]:
fdr_mask = sr_qval < 0.05

# Figure 1: Spearman Rho (thresholded at p < .05)
plot_heatmap(
    sr_stat_sig, row_labels, pet_labels,
    cmap="RdBu_r", clim=(-0.4, 0.4),
    cbar_label="Spearman \u03c1",
    title="Spearman \u03c1 (thresholded at uncorrected p < .05)",
    sig_mask=fdr_mask,
    fname="cog_ep_anat_receptor_rho.svg", figs_dir=figs_dir
)

In [ ]:
# Figure 2: P-values (spin test)
plot_heatmap(
    sr_pval, row_labels, pet_labels,
    cmap="Reds_r", clim=(0, 0.05),
    cbar_label="p-value",
    title="P-values (spin permutation test)",
    sig_mask=fdr_mask,
    fname="cog_ep_anat_receptor_pval.svg", figs_dir=figs_dir
)

In [ ]:
# Figure 3: Q-values (FDR)
plot_heatmap(
    sr_qval, row_labels, pet_labels,
    cmap="Reds_r", clim=(0, 0.05),
    cbar_label="q-value",
    title="FDR-corrected Q-values (Benjamini-Hochberg)",
    sig_mask=fdr_mask,
    fname="cog_ep_anat_receptor_qval.svg", figs_dir=figs_dir
)

## Summary Table

In [ ]:
# Print all significant associations (FDR q < 0.05)
print("Significant receptor-domain associations (FDR q < .05):\n")
print(f"{'Domain':<12} {'Receptor':<10} {'Rho':>8} {'p-value':>10} {'q-value':>10}")
print("-" * 52)

for i, label in enumerate(row_labels):
    for j, receptor in enumerate(pet_labels):
        if sr_qval[i, j] < 0.05:
            print(f"{label:<12} {receptor:<10} {sr_stat[i, j]:>8.3f} {sr_pval[i, j]:>10.4f} {sr_qval[i, j]:>10.4f}")

In [ ]:
# All associations with uncorrected p < .05
print("\nAll associations with uncorrected p < .05:\n")
print(f"{'Domain':<12} {'Receptor':<10} {'Rho':>8} {'p-value':>10} {'q-value':>10}")
print("-" * 52)

for i, label in enumerate(row_labels):
    for j, receptor in enumerate(pet_labels):
        if sr_pval[i, j] < 0.05:
            fdr_flag = "*" if sr_qval[i, j] < 0.05 else ""
            print(f"{label:<12} {receptor:<10} {sr_stat[i, j]:>8.3f} {sr_pval[i, j]:>10.4f} {sr_qval[i, j]:>10.4f} {fdr_flag}")